# 2D Gaussian Reconstruction
### Based on 3DGR-CT: Sparse-View CT Reconstruction with a 3D Gaussian Representation (Li et al., 2025)
**TRDP Work 2** — 2D implementation of Gaussian-based CT reconstruction across three datasets and four acquisition geometries.

---

**Method overview:**
Each 2D Gaussian Gᵢ is defined by:
- **μᵢ** = (x, y) — center position
- **Σᵢ** = R S Sᵀ Rᵀ — covariance (rotation + anisotropic scaling)
- **tᵢ** — intensity (CT attenuation value)

The image volume is: V(X) = Σᵢ tᵢ · exp(−½ (X-μᵢ)ᵀ Σᵢ⁻¹ (X-μᵢ))

Optimization minimizes: ‖P̂(V({θᵢ})) - P‖²₂  (projection-space L2 loss)

## Install Dependencies

In [ ]:
!pip install numpy matplotlib scikit-image torch astra-toolbox pydicom -q

## Core Imports & Global Configuration

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import astra
import pydicom
from pydicom.data import get_testdata_file
from skimage.data import shepp_logan_phantom
from skimage.transform import resize
from skimage.draw import rectangle
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import random # Added for global random seed

# Function to set global random seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Global Configurations
VOL_SIZE       = 256
DETECTOR_COUNT = 384
ANGLES_FULL    = 180

full_angles_ref = np.linspace(0, np.pi, ANGLES_FULL, endpoint=False)
vol_geom        = astra.create_vol_geom(VOL_SIZE, VOL_SIZE)

HAS_GPU = torch.cuda.is_available()
DEVICE  = torch.device('cuda' if HAS_GPU else 'cpu')
print(f"CUDA Available : {HAS_GPU}")
print(f"Device         : {DEVICE}")
if HAS_GPU:
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

# Set a global seed for reproducibility
set_seed(42)

## Shared Utility Functions
Metric calculation and sinogram generation — reused from Work 1 conventions.

In [ ]:
# ── Metrics ─────────────────────────────────────────────────────────────────
def calculate_metrics(gt, recon):
    """Compute PSNR and SSIM; normalises reconstruction to [0,1] first."""
    recon_norm = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    val_psnr = psnr(gt, recon_norm, data_range=1.0)
    val_ssim = ssim(gt, recon_norm, data_range=1.0)
    return val_psnr, val_ssim


# ── Sinogram generation via ASTRA ────────────────────────────────────────────
def generate_sinogram(image, angles):
    """
    Forward-project `image` at the given angles using ASTRA parallel-beam.
    Returns sinogram as a numpy array of shape (len(angles), DETECTOR_COUNT).
    """
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    proj_id   = astra.create_projector('linear', proj_geom, vol_geom)
    sino_id, sinogram = astra.create_sino(image, proj_id)
    sinogram = sinogram.copy()
    astra.data2d.delete(sino_id)
    astra.projector.delete(proj_id)
    return sinogram


# ── Angle set definitions (identical to Work 1) ──────────────────────────────
def get_angle_sets():
    """Return the four acquisition geometry angle arrays."""
    np.random.seed(42)
    angles_full    = np.linspace(0, np.pi, 180, endpoint=False)
    angles_uniform = np.linspace(0, np.pi, 20,  endpoint=False)
    angles_random  = np.sort(np.random.choice(full_angles_ref, 20, replace=False))
    angles_wedge   = np.concatenate((
        np.linspace(0,          np.pi/3, 60, endpoint=False),
        np.linspace(2*np.pi/3, np.pi,   60, endpoint=False)
    ))
    return {
        'Full View'          : angles_full,
        'Uniform Sparse View': angles_uniform,
        'Random Sparse View' : angles_random,
        'Missing Wedge'      : angles_wedge,
    }

print("Shared utilities loaded.")

## Dataset Loaders
Three datasets matching Work 1: Shepp-Logan phantom, Checkerboard, and DICOM.

In [ ]:
def load_shepp_logan():
    """Load and normalise the Shepp-Logan phantom to VOL_SIZE × VOL_SIZE."""
    img = shepp_logan_phantom()
    img = resize(img, (VOL_SIZE, VOL_SIZE), anti_aliasing=True)
    img = (img - img.min()) / (img.max() - img.min())
    return img.astype(np.float32)


def create_checkerboard(size=VOL_SIZE, num_squares=8):
    """Create a binary checkerboard image."""
    img  = np.zeros((size, size), dtype=np.float32)
    step = size // num_squares
    for y in range(0, size, step):
        for x in range(0, size, step):
            if (x // step + y // step) % 2 == 1:
                rr, cc = rectangle((y, x), extent=(step, step), shape=img.shape)
                img[rr, cc] = 1.0
    return img


def load_dicom():
    """Load the built-in pydicom CT test file, resize, and normalise."""
    filename = get_testdata_file("CT_small.dcm")
    dataset  = pydicom.dcmread(filename)
    img      = dataset.pixel_array.astype(np.float64)
    img      = resize(img, (VOL_SIZE, VOL_SIZE), anti_aliasing=True)
    img      = (img - img.min()) / (img.max() - img.min())
    return img.astype(np.float32)


# Pre-load all three datasets
datasets = {
    'Shepp-Logan'  : load_shepp_logan(),
    'Checkerboard' : create_checkerboard(),
    'DICOM'        : load_dicom(),
}
print("Datasets loaded:")
for name, img in datasets.items():
    print(f"  {name:15s}  shape={img.shape}  min={img.min():.3f}  max={img.max():.3f}")

---
## 2D Gaussian Representation

### `Gaussians2D` — Core Representation Class

Each Gaussian Gᵢ carries learnable parameters stored as `torch.nn.Parameter`:
| Parameter | Shape | Meaning |
|-----------|-------|---------|
| `mu`      | (N,2) | Centre (x, y) in [0,1] normalised space |
| `log_s`   | (N,2) | Log-scale (→ exp gives sx, sy > 0) |
| `theta`   | (N,1) | Rotation angle in radians |
| `t_raw`   | (N,1) | Pre-sigmoid intensity |

Covariance decomposition: Σ = R S Sᵀ Rᵀ, R = [[cosθ, −sinθ],[sinθ, cosθ]], S = diag(sx, sy)

In [ ]:
class Gaussians2D(torch.nn.Module):
    """
    Learnable 2D Gaussian field for CT reconstruction.

    Parameters (all torch.nn.Parameter):
        mu     (N, 2)  — centres in normalised [0,1] space
        log_s  (N, 2)  — log-scale → exp gives sx, sy > 0
        theta  (N, 1)  — rotation angle in radians
        t_raw  (N, 1)  — pre-sigmoid intensity

    Buffers (non-parameter, not differentiated):
        grad_accum       (N,) — accumulated |∇μ| over ADC window
        grad_accum_steps int  — number of steps accumulated so far
        _flat_idx, _gauss_ids, _cache_grid, _index_dirty — index cache
    """

    def __init__(self, mu_init, s_init, t_init, device=DEVICE):
        super().__init__()
        N = mu_init.shape[0]
        self.mu    = torch.nn.Parameter(torch.tensor(mu_init,        dtype=torch.float32, device=device))
        self.log_s = torch.nn.Parameter(torch.tensor(np.log(s_init), dtype=torch.float32, device=device))
        self.theta = torch.nn.Parameter(torch.zeros(N, 1,            dtype=torch.float32, device=device))
        self.t_raw = torch.nn.Parameter(torch.tensor(
            np.log(np.clip(t_init, 1e-6, 1-1e-6) / (1.0 - np.clip(t_init, 1e-6, 1-1e-6))).reshape(-1, 1),
            dtype=torch.float32, device=device))

        # ── Gradient accumulation buffer (for ADC densification criterion) ──
        # Mirrors the 3DGS paper approach: accumulate |∇μ| over the densification
        # window, then use the mean as the clone/split trigger.
        self.grad_accum       = torch.zeros(N, device=device)  # (N,)
        self.grad_accum_steps = 0

        # ── Index cache ─────────────────────────────────────────────────
        self._flat_idx    = None
        self._gauss_ids   = None
        self._cache_grid  = None
        self._index_dirty = True

    def accumulate_grad(self):
        """
        Called after every loss.backward(). Accumulates the L2 norm of
        the raw positional gradient for each Gaussian.
        Safe to call even if mu.grad is None (no-op).
        """
        if self.mu.grad is not None:
            with torch.no_grad():
                self.grad_accum += self.mu.grad.norm(dim=1)
            self.grad_accum_steps += 1

    def reset_grad_accum(self):
        """Zero the accumulation buffer. Called after each ADC pass."""
        self.grad_accum.zero_()
        self.grad_accum_steps = 0

    # ── Activations ───────────────────────────────────────────────────
    @property
    def scales(self):
        return torch.exp(self.log_s)

    @property
    def intensities(self):
        return torch.sigmoid(self.t_raw)

    # ── Covariance inverse ────────────────────────────────────────────
    def get_cov_inv(self):
        """Σ⁻¹ = R diag(1/sx², 1/sy²) Rᵀ, shape (N, 2, 2)."""
        s   = self.scales
        cos = torch.cos(self.theta)
        sin = torch.sin(self.theta)

        si0_sq = 1.0 / (s[:, 0:1] ** 2 + 1e-8)
        si1_sq = 1.0 / (s[:, 1:2] ** 2 + 1e-8)

        c00 = cos**2 * si0_sq + sin**2 * si1_sq
        c01 = cos * sin * (si0_sq - si1_sq)
        c11 = sin**2 * si0_sq + cos**2 * si1_sq

        return torch.stack([
            torch.cat([c00, c01], dim=1),
            torch.cat([c01, c11], dim=1),
        ], dim=1)

    # ── Index cache builder ────────────────────────────────────────────
    def _rebuild_index_cache(self, grid_size):
        """
        Build flat_idx (M,) and gauss_ids (M,) for all (Gaussian, pixel)
        pairs within 3σ. Called when _index_dirty=True or grid_size changes.
        Uses group-by-radius vectorisation to avoid N=8000 Python iterations.
        """
        device = self.mu.device
        G      = grid_size
        N      = self.mu.shape[0]

        with torch.no_grad():
            mu_px  = (self.mu * (G - 1)).round().long().clamp(0, G - 1)
            cx_np  = mu_px[:, 0].cpu().numpy().astype(np.int32)
            cy_np  = mu_px[:, 1].cpu().numpy().astype(np.int32)
            sig_px = self.scales.max(dim=1).values * (G - 1)
            r_np   = sig_px.mul(3.0).ceil().long().clamp(min=1).cpu().numpy().astype(np.int32)

        flat_list = []
        gids_list = []

        for r in np.unique(r_np):
            group  = np.where(r_np == r)[0].astype(np.int32)
            side   = 2 * r + 1
            off    = np.arange(-r, r + 1, dtype=np.int32)
            ro, co = np.meshgrid(off, off, indexing='ij')
            ro = ro.ravel(); co = co.ravel()

            rows = cy_np[group, None] + ro[None, :]
            cols = cx_np[group, None] + co[None, :]

            valid = (rows >= 0) & (rows < G) & (cols >= 0) & (cols < G)
            rows  = np.clip(rows, 0, G - 1)
            cols  = np.clip(cols, 0, G - 1)
            flat  = rows * G + cols
            gids  = np.broadcast_to(group[:, None], flat.shape)

            flat_list.append(flat[valid])
            gids_list.append(gids[valid])

        flat_np = np.concatenate(flat_list).astype(np.int64)
        gids_np = np.concatenate(gids_list).astype(np.int64)

        self._flat_idx    = torch.from_numpy(flat_np).to(device)
        self._gauss_ids   = torch.from_numpy(gids_np).to(device)
        self._cache_grid  = grid_size
        self._index_dirty = False

    # ── Voxelisation ──────────────────────────────────────────────────
    def voxelise(self, grid_size=VOL_SIZE):
        """
        Rasterise all Gaussians onto a (grid_size × grid_size) image.
        Uses cached flat indices and a single scatter_add_ for efficiency.
        """
        device = self.mu.device
        G      = grid_size

        if self._index_dirty or self._cache_grid != G:
            self._rebuild_index_cache(G)

        flat_idx  = self._flat_idx
        gauss_ids = self._gauss_ids

        row_idx = flat_idx // G
        col_idx = flat_idx %  G

        px_x = col_idx.float() / (G - 1)
        px_y = row_idx.float() / (G - 1)

        mu_m  = self.mu[gauss_ids]
        ci_m  = self.get_cov_inv()[gauss_ids]
        t_m   = self.intensities.squeeze(1)[gauss_ids]

        dx   = px_x - mu_m[:, 0]
        dy   = px_y - mu_m[:, 1]
        diff = torch.stack([dx, dy], dim=1).unsqueeze(2)

        D2   = torch.bmm(torch.bmm(diff.transpose(1, 2), ci_m), diff).squeeze(-1).squeeze(-1)
        vals = t_m * torch.exp(-0.5 * D2)

        volume = torch.zeros(G * G, dtype=torch.float32, device=device)
        volume.scatter_add_(0, flat_idx, vals)

        return volume.view(G, G)


print("Gaussians2D (with grad_accum buffer) defined.")

### FBP-Guided Gaussian Initialisation
Replicates Section 3.1 of 3DGR-CT: centres placed at medium-gradient voxels of the FBP image, covariance set inversely proportional to local density, intensity proportional to FBP value.

In [ ]:
def fbp_init_gaussians(fbp_image, n_gaussians=8000, tau=0.05,
                       k_sigma=0.25, k_t=0.5, neighbor_radius=5):
    """
    FBP-image-guided Gaussian initialisation (Section 3.1 of 3DGR-CT).

    Changes vs previous version:
        Step 3: Uniform foreground sampling replaces medium-gradient filter.
            The medium-gradient percentile heuristic was designed for dense-view
            (180-angle) FBP where streaks are negligible. With 20-angle FBP the
            30th-70th gradient percentile selects streak pixels, which lie in a
            regular grid pattern. This grid is then cloned by ADC into 40,000
            grid points, producing interference artifacts in the reconstruction.
            Uniform sampling avoids this entirely.
        Step 3b: 1-pixel Gaussian jitter breaks residual grid symmetry.
        k_t: 0.15 -> 0.5 for stronger initial signal.
        sigma clip max: 0.05 -> 0.015 (3.8px radius, prevents over-coverage).

    Args:
        fbp_image      : 2D numpy array, normalised [0,1]
        n_gaussians    : target number of Gaussians
        tau            : background threshold
        k_sigma        : scale coefficient for covariance init
        k_t            : intensity multiplier on FBP values
        neighbor_radius: radius (pixels) for neighbour density estimate

    Returns:
        mu_init : (N, 2) float32 — centres in [0,1]
        s_init  : (N, 2) float32 — isotropic scales
        t_init  : (N,)   float32 — intensities
    """
    H, W = fbp_image.shape

    # ── Step 1: threshold to remove background ───────────────────────────
    foreground = fbp_image > tau

    # ── Step 2: compute gradient magnitude (kept for sigma init only) ────
    gy, gx   = np.gradient(fbp_image)
    grad_mag = np.sqrt(gx**2 + gy**2)
    grad_mag[~foreground] = 0.0

    # ── Step 3: UNIFORM foreground sampling ──────────────────────────────
    # Do NOT filter by gradient percentile. In sparse-view FBP (20 angles),
    # the 30th-70th percentile selects streak artifacts which lie on a
    # regular grid. Instead sample uniformly from all foreground pixels.
    candidate_ys, candidate_xs = np.where(foreground)

    if len(candidate_ys) == 0:
        # Fallback: uniform random if foreground mask is empty
        candidate_ys = np.random.randint(0, H, n_gaussians)
        candidate_xs = np.random.randint(0, W, n_gaussians)

    n_cands = len(candidate_ys)
    idx     = np.random.choice(n_cands, n_gaussians,
                               replace=(n_cands < n_gaussians))
    ys, xs  = candidate_ys[idx], candidate_xs[idx]

    N = len(ys)

    # ── Step 3b: add 1-pixel Gaussian jitter ─────────────────────────────
    # Breaks any residual periodicity from the foreground mask geometry.
    jitter_sigma = 1.5 / max(H, W)  # 1.5 pixels in normalised coords
    jitter_x = np.random.normal(0.0, jitter_sigma, N)
    jitter_y = np.random.normal(0.0, jitter_sigma, N)

    # ── Step 4: normalise centres to [0,1] ──────────────────────────────
    mu_x    = np.clip(xs / (W - 1) + jitter_x, 0.0, 1.0)
    mu_y    = np.clip(ys / (H - 1) + jitter_y, 0.0, 1.0)
    mu_init = np.stack([mu_x, mu_y], axis=1).astype(np.float32)

    # ── Step 5: neighbour-density covariance init ────────────────────────
    # σ = k_sigma / N(C, r) — denser regions get smaller Gaussians.
    # Clip max = 0.015 (~3.8px) prevents over-coverage; min = 5e-4.
    from scipy.spatial import cKDTree
    tree        = cKDTree(mu_init)
    n_neighbors = np.array(
        [len(tree.query_ball_point(mu_init[i], r=neighbor_radius / (W - 1)))
         for i in range(N)], dtype=np.float32)
    n_neighbors = np.maximum(n_neighbors, 1.0)
    sigma_vals  = k_sigma / n_neighbors
    sigma_vals  = np.clip(sigma_vals, 5e-4, 0.015)
    s_init      = np.stack([sigma_vals, sigma_vals], axis=1).astype(np.float32)

    # ── Step 6: intensity from FBP image ─────────────────────────────────
    t_raw  = fbp_image[ys, xs].astype(np.float32)
    t_init = np.clip(k_t * t_raw, 1e-4, 0.9999)

    return mu_init, s_init, t_init


print("FBP-guided initialisation defined.")

### Differentiable CT Projector
Wraps ASTRA's forward projection inside a `torch.autograd.Function` so gradients flow back through the sinogram into the Gaussian parameters. This is the 2D equivalent of the differentiable CT projector in Section 3.2 of 3DGR-CT.

In [ ]:
class DifferentiableCTProjector(torch.autograd.Function):
    """
    Differentiable parallel-beam CT projector backed by ASTRA.

    Forward  : volume (H×W tensor) → sinogram (A×D tensor)
    Backward : grad_sinogram        → grad_volume via BP_CUDA / BP

    Fix vs previous version: backward now correctly selects BP_CUDA
    when a GPU is available, avoiding the silent CPU fallback that was
    making every backward pass ~100× slower than the forward.
    """

    @staticmethod
    def forward(ctx, volume_tensor, angles):
        ctx.angles = angles
        ctx.vol_size = volume_tensor.shape[-1] # Store volume size
        device     = volume_tensor.device

        vol_np  = volume_tensor.detach().cpu().numpy().astype(np.float32)
        sino_np = _astra_fp(vol_np, angles)

        return torch.tensor(sino_np, dtype=torch.float32, device=device)

    @staticmethod
    def backward(ctx, grad_sinogram):
        angles = ctx.angles
        vol_size = ctx.vol_size # Retrieve volume size
        device = grad_sinogram.device

        grad_sino_np = grad_sinogram.detach().cpu().numpy().astype(np.float32)
        grad_vol_np  = _astra_bp(grad_sino_np, angles, vol_size) # Pass vol_size

        return torch.tensor(grad_vol_np, dtype=torch.float32, device=device), None


def _astra_fp(vol_np, angles):
    """ASTRA forward projection (FP_CUDA when available)."""
    H, W      = vol_np.shape
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    vol_geom_ = astra.create_vol_geom(H, W)

    # Use CUDA projector when available — silent CPU fallback was the bug
    if HAS_GPU:
        vol_id  = astra.data2d.create('-vol',  vol_geom_,  vol_np)
        sino_id = astra.data2d.create('-sino', proj_geom,  0)
        cfg     = astra.astra_dict('FP_CUDA')
        cfg['VolumeDataId']     = vol_id
        cfg['ProjectionDataId'] = sino_id
        alg_id  = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        sino = astra.data2d.get(sino_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(sino_id)
        astra.data2d.delete(vol_id)
    else:
        proj_id = astra.create_projector('linear', proj_geom, vol_geom_)
        sino_id, sino = astra.create_sino(vol_np, proj_id)
        sino = sino.copy()
        astra.data2d.delete(sino_id)
        astra.projector.delete(proj_id)

    return sino.astype(np.float32)


def _astra_bp(grad_sino_np, angles, vol_size): # Accept vol_size
    """ASTRA back-projection adjoint (BP_CUDA when available)."""
    proj_geom = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    vol_geom_ = astra.create_vol_geom(vol_size, vol_size) # Use vol_size here

    if HAS_GPU:
        sino_id = astra.data2d.create('-sino', proj_geom,  grad_sino_np)
        rec_id  = astra.data2d.create('-vol',  vol_geom_,  0.0)
        cfg     = astra.astra_dict('BP_CUDA')
        cfg['ReconstructionDataId'] = rec_id
        cfg['ProjectionDataId']     = sino_id
        alg_id  = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        grad_vol = astra.data2d.get(rec_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(rec_id)
        astra.data2d.delete(sino_id)
    else:
        sino_id  = astra.data2d.create('-sino', proj_geom,  grad_sino_np)
        rec_id   = astra.data2d.create('-vol',  vol_geom_,  0.0)
        proj_id  = astra.create_projector('linear', proj_geom, vol_geom_)
        cfg      = astra.astra_dict('BP')
        cfg['ReconstructionDataId'] = rec_id
        cfg['ProjectionDataId']     = sino_id
        cfg['ProjectorId']          = proj_id
        alg_id   = astra.algorithm.create(cfg)
        astra.algorithm.run(alg_id)
        grad_vol = astra.data2d.get(rec_id).copy()
        astra.algorithm.delete(alg_id)
        astra.data2d.delete(rec_id)
        astra.data2d.delete(sino_id)
        astra.projector.delete(proj_id)

    return grad_vol.astype(np.float32)


def project(volume_tensor, angles):
    """
    Differentiable entry point.
    volume_tensor : (H, W) float32 torch tensor
    angles        : numpy array of projection angles
    Returns       : (A, D) sinogram tensor, same device
    """
    return DifferentiableCTProjector.apply(volume_tensor, angles)


print("Differentiable CT projector (FP_CUDA / BP_CUDA) defined.")

### Adaptive Density Control
Clone / Split / Prune operations from Section 3.3 of 3DGR-CT, adapted to 2D.
| Operation | Trigger | Action |
|-----------|---------|--------|
| **Clone** | High ‖∇μ‖, small scale | Duplicate; one moves along gradient |
| **Split** | High ‖∇μ‖, large scale | Replace with 2 smaller Gaussians (x0.8 scale, ÷2 intensity) |
| **Prune** | Near-zero intensity | Remove |

In [ ]:
def adaptive_density_control(gaussians, densify_percentile=95,
                              prune_threshold=1e-6,
                              sigma_split_thresh=0.01,
                              max_gaussians=12000):
    """
    Clone / split / prune Gaussians.

    Changes from V4:
        prune_threshold: 0.005 -> 1e-6
            The old threshold of 0.005 (0.5% intensity) was pruning interior
            Gaussians that represent low-contrast brain tissue (typical intensity
            0.01-0.04). These Gaussians have legitimate intensity but drift just
            below threshold from optimiser noise, causing the steady PSNR decline
            from 19.68 dB (iter 1000) to 17.16 dB (iter 7999) as the interior
            is progressively emptied. Only truly invisible Gaussians (< 1e-6)
            are now pruned.

        sigma_split_thresh: 0.01 (NEW, replaces relative median criterion)
            The old split criterion (scale >= median(scale)) became meaningless
            once the population homogenised in scale. After three ADC calls,
            all Gaussians had similar scale, the high-gradient ones happened to
            fall below the median, and splits dropped to zero from iter 1200
            onward. An absolute threshold of 0.01 (~2.5px at 256px) has clear
            physical meaning: only Gaussians larger than 2.5px radius are split.
            When all are below 2.5px, splits stop naturally — which is correct.

    Returns:
        new_gaussians : Gaussians2D
        source_idx    : (N_new,) LongTensor — maps new -> old indices
        stats         : dict with clone/split/prune counts
    """
    device = gaussians.mu.device

    with torch.no_grad():
        mu    = gaussians.mu.data
        log_s = gaussians.log_s.data
        theta = gaussians.theta.data
        t_raw = gaussians.t_raw.data
        s     = gaussians.scales
        t     = gaussians.intensities
        N     = mu.shape[0]

        # ── Percentile-based densification threshold ──────────────────────
        if gaussians.mu.grad is not None:
            grad_norm  = gaussians.mu.grad.norm(dim=1)
            pct_thresh = torch.quantile(grad_norm, densify_percentile / 100.0)
            high_grad  = grad_norm > pct_thresh
        else:
            high_grad = torch.zeros(N, dtype=torch.bool, device=device)

        mean_scale = s.mean(dim=1)

        # ── Absolute split threshold (replaces relative median) ───────────
        # Split if the Gaussian is both high-gradient AND physically large.
        # sigma_split_thresh=0.01 ≈ 2.5 pixels at 256px resolution.
        # Stops naturally when all Gaussians have been reduced below this size.
        large_s = mean_scale > sigma_split_thresh

        # ── Prune: only truly invisible Gaussians ─────────────────────────
        low_t = t.squeeze(1) < prune_threshold   # prune_threshold=1e-6

        clone_mask = high_grad & ~large_s & ~low_t   # small + high grad -> clone
        split_mask = high_grad &  large_s & ~low_t   # large + high grad -> split
        keep_mask  = ~low_t                           # keep everything visible

        keep_idx  = torch.where(keep_mask)[0]
        clone_idx = torch.where(clone_mask)[0]
        split_idx = torch.where(split_mask)[0]

        n_clone = clone_idx.shape[0]
        n_split = split_idx.shape[0]
        n_prune = low_t.sum().item()

        mu_new    = [mu[keep_mask]]
        log_s_new = [log_s[keep_mask]]
        theta_new = [theta[keep_mask]]
        t_raw_new = [t_raw[keep_mask]]
        source_parts = [keep_idx]

        # ── Clone ─────────────────────────────────────────────────────────
        if clone_mask.any():
            mu_c     = mu[clone_mask]
            grad_c   = gaussians.mu.grad[clone_mask]
            grad_dir = grad_c / (grad_c.norm(dim=1, keepdim=True) + 1e-8)
            mu_moved = torch.clamp(mu_c + 0.01 * grad_dir, 0.0, 1.0)
            mu_new    += [mu_moved]
            log_s_new += [log_s[clone_mask]]
            theta_new += [theta[clone_mask]]
            t_raw_new += [t_raw[clone_mask]]
            source_parts.append(clone_idx)

        # ── Split ─────────────────────────────────────────────────────────
        if split_mask.any():
            mu_sp     = mu[split_mask]
            s_sp      = s[split_mask]
            new_log_s = log_s[split_mask] + np.log(0.8)   # scale × 0.8
            t_half    = torch.clamp(
                torch.sigmoid(t_raw[split_mask]) / 2.0, 1e-6, 1 - 1e-6)
            new_t_raw = torch.log(t_half / (1.0 - t_half))
            noise     = torch.randn_like(mu_sp) * s_sp * 0.3
            mu_new    += [torch.clamp(mu_sp + noise, 0.0, 1.0),
                          torch.clamp(mu_sp - noise, 0.0, 1.0)]
            log_s_new += [new_log_s, new_log_s]
            theta_new += [theta[split_mask]] * 2
            t_raw_new += [new_t_raw, new_t_raw]
            source_parts += [split_idx, split_idx]

        mu_cat     = torch.cat(mu_new,    dim=0)
        log_s_cat  = torch.cat(log_s_new, dim=0)
        theta_cat  = torch.cat(theta_new, dim=0)
        t_raw_cat  = torch.cat(t_raw_new, dim=0)
        source_idx = torch.cat(source_parts, dim=0)

        if mu_cat.shape[0] > max_gaussians:
            perm       = torch.randperm(mu_cat.shape[0], device=device)[:max_gaussians]
            mu_cat     = mu_cat[perm]
            log_s_cat  = log_s_cat[perm]
            theta_cat  = theta_cat[perm]
            t_raw_cat  = t_raw_cat[perm]
            source_idx = source_idx[perm]

    # ── Build new Gaussians2D ─────────────────────────────────────────────
    new_g = Gaussians2D.__new__(Gaussians2D)
    torch.nn.Module.__init__(new_g)
    new_g.mu    = torch.nn.Parameter(mu_cat.to(device))
    new_g.log_s = torch.nn.Parameter(log_s_cat.to(device))
    new_g.theta = torch.nn.Parameter(theta_cat.to(device))
    new_g.t_raw = torch.nn.Parameter(t_raw_cat.to(device))
    new_g._flat_idx    = None
    new_g._gauss_ids   = None
    new_g._cache_grid  = None
    new_g._index_dirty = True

    stats = {'clone': n_clone, 'split': n_split, 'prune': n_prune,
             'N_new': mu_cat.shape[0]}
    return new_g, source_idx, stats


print("ADC V5 (prune_threshold=1e-6, absolute split threshold sigma=0.01) defined.")

### Two-Stage Optimiser & Training Loop
Mirrors Section 3.2 of 3DGR-CT:
- **Stage 1** (low-resolution 128x128): fast coarse placement
- **Stage 2** (full-resolution 256x256): fine-grained refinement

Learning rates per parameter type match the paper exactly.

In [ ]:
def make_optimizer(gaussians):
    """
    Create a fresh Adam optimizer with initial learning rates.
    All four groups are decayed each iteration via compute_all_lrs().
    """
    return torch.optim.Adam([
        {'params': [gaussians.mu],    'lr': 2e-3,  'name': 'mu'},
        {'params': [gaussians.log_s], 'lr': 5e-3,  'name': 'log_s'},
        {'params': [gaussians.theta], 'lr': 1e-3,  'name': 'theta'},
        {'params': [gaussians.t_raw], 'lr': 5e-2,  'name': 't_raw'},
    ], betas=(0.9, 0.999))


def compute_all_lrs(iteration, n_iters):
    """
    Exponential LR decay for all four parameter groups.
    Each group decays by a factor of 1000x over n_iters iterations.

    Returns a dict keyed by the 'name' field in each optimizer param_group.
    In the training loop, apply with:
        for group in optimizer.param_groups:
            group['lr'] = all_lrs[group['name']]
    """
    def decay(start, end):
        return start * (end / start) ** (iteration / n_iters)

    return {
        'mu':    decay(2e-3, 2e-6),
        'log_s': decay(5e-3, 5e-5),
        'theta': decay(1e-3, 1e-5),
        't_raw': decay(5e-2, 5e-4),
    }


def update_optimizer_state(old_optimizer, new_optimizer, source_idx):
    """
    Transfer Adam momentum from old → new optimizer after ADC.
    source_idx[i] gives the old Gaussian index that new Gaussian i descended from.
    """
    for old_group, new_group in zip(old_optimizer.param_groups,
                                    new_optimizer.param_groups):
        old_param = old_group['params'][0]
        new_param = new_group['params'][0]

        if old_param not in old_optimizer.state:
            continue

        old_state = old_optimizer.state[old_param]
        new_state = new_optimizer.state[new_param]

        new_state['step'] = old_state['step']

        for key in ('exp_avg', 'exp_avg_sq'):
            if key in old_state:
                new_state[key] = old_state[key][source_idx].clone()

    return new_optimizer


print("make_optimizer, compute_all_lrs, update_optimizer_state defined.")

In [ ]:
def train_gaussian_reconstruction(
    image_np,
    angles,
    n_gaussians        = 8000,
    n_iters            = 8000,
    adc_start          = 500,
    adc_every          = 200,
    adc_stop           = 4000,
    densify_percentile = 95,
    prune_thresh       = 1e-6,
    max_gaussians      = 12000,
    verbose            = True,
):
    """
    Single-stage 2D Gaussian CT reconstruction — V5.

    Changes from V4:
        prune_thresh: 0.005 -> 1e-6
            Passed to adaptive_density_control. Prevents interior Gaussians
            (brain tissue, intensity ~0.01-0.04) from being pruned.

        adc_stop: 6000 -> 4000
            ADC ran until iter 6000 in V4, during which PSNR fell from 18.5 to
            17.1 dB (ADC churn degrading a near-converged model). Stopping at
            iter 4000 gives 4000 uninterrupted refinement iterations instead of
            2000, at a point where the model still has significant room to improve.
    """
    device = DEVICE

    # ── Ground-truth sinogram ─────────────────────────────────────────────
    sino_gt_np = generate_sinogram(image_np, angles)
    sino_gt    = torch.tensor(sino_gt_np, dtype=torch.float32, device=device)

    # ── FBP initialisation ────────────────────────────────────────────────
    proj_geom   = astra.create_proj_geom('parallel', 1.0, DETECTOR_COUNT, angles)
    proj_id_fbp = astra.create_projector('linear', proj_geom, vol_geom)
    sino_id_fbp, _ = astra.create_sino(image_np, proj_id_fbp)

    rec_id_fbp = astra.data2d.create('-vol', vol_geom, 0.0)
    cfg_fbp    = astra.astra_dict('FBP_CUDA' if HAS_GPU else 'FBP')
    cfg_fbp['ReconstructionDataId'] = rec_id_fbp
    cfg_fbp['ProjectionDataId']     = sino_id_fbp
    if not HAS_GPU:
        cfg_fbp['ProjectorId'] = proj_id_fbp
    alg_fbp = astra.algorithm.create(cfg_fbp)
    astra.algorithm.run(alg_fbp)
    fbp_image = astra.data2d.get(rec_id_fbp).copy()
    astra.algorithm.delete(alg_fbp)
    astra.data2d.delete(rec_id_fbp)
    astra.data2d.delete(sino_id_fbp)
    astra.projector.delete(proj_id_fbp)

    fbp_image = (fbp_image - fbp_image.min()) / \
                (fbp_image.max() - fbp_image.min() + 1e-8)
    fbp_image = fbp_image.astype(np.float32)

    mu_init, s_init, t_init = fbp_init_gaussians(fbp_image,
                                                   n_gaussians=n_gaussians)
    gaussians = Gaussians2D(mu_init, s_init, t_init, device=device)
    optimizer = make_optimizer(gaussians)

    loss_curve = []
    psnr_curve = []
    start_time = time.time()

    for iteration in range(n_iters):

        # ── Decay ALL parameter group learning rates ───────────────────────
        all_lrs = compute_all_lrs(iteration, n_iters)
        for group in optimizer.param_groups:
            group['lr'] = all_lrs[group['name']]

        # ── Forward + backward ────────────────────────────────────────────
        optimizer.zero_grad()
        volume    = gaussians.voxelise(grid_size=VOL_SIZE)
        sino_pred = project(volume, angles)
        loss      = torch.mean((sino_pred - sino_gt) ** 2)
        loss.backward()

        all_params = [gaussians.mu, gaussians.log_s,
                      gaussians.theta, gaussians.t_raw]
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        optimizer.step()

        # ── ADC (disabled after adc_stop) ─────────────────────────────────
        adc_due = (iteration >= adc_start
                   and iteration <= adc_stop
                   and (iteration % adc_every) == 0)

        if adc_due:
            old_optimizer = optimizer
            gaussians, source_idx, stats = adaptive_density_control(
                gaussians,
                densify_percentile=densify_percentile,
                prune_threshold=prune_thresh,
                max_gaussians=max_gaussians,
            )
            optimizer = make_optimizer(gaussians)
            optimizer = update_optimizer_state(old_optimizer, optimizer,
                                               source_idx)
            for group in optimizer.param_groups:
                group['lr'] = all_lrs[group['name']]

            if verbose:
                print(f"    [ADC iter {iteration}] "
                      f"clone={stats['clone']}  split={stats['split']}  "
                      f"prune={stats['prune']}  N={stats['N_new']}")

        # ── Logging ───────────────────────────────────────────────────────
        loss_val = loss.item()
        loss_curve.append(loss_val)

        with torch.no_grad():
            recon_display = volume.detach().cpu().numpy()
            p, _ = calculate_metrics(image_np, recon_display)
            psnr_curve.append(p)

        if verbose and (iteration % 500 == 0 or iteration == n_iters - 1):
            phase = 'refine' if iteration > adc_stop else 'densify'
            print(f"  Iter {iteration:5d}/{n_iters}  [{phase}]  "
                  f"Loss={loss_val:.6f}  PSNR={p:.2f} dB  "
                  f"N={gaussians.mu.shape[0]}")

    with torch.no_grad():
        recon_np = gaussians.voxelise(grid_size=VOL_SIZE).cpu().numpy()

    return recon_np, loss_curve, psnr_curve, time.time() - start_time, gaussians


print("Training loop V5 (prune_thresh=1e-6, adc_stop=4000) defined.")

### Visualisation Utilities
`plot_gaussian_experiment` — matches the 4-panel layout of Work 1's `plot_experiment` and adds a convergence curve panel.

In [ ]:
# ── Per-dataset ROI crop definitions ────────────────────────────────────────
# Each entry is (y0, y1, x0, x1) in pixel coords on the 256×256 image.
# Chosen to capture the most diagnostically relevant high-frequency structure
# in each dataset, so the zoomed panel shows where the method succeeds or fails.
#
#   Shepp-Logan  : 80×80 crop of the upper-right skull quadrant — the elliptical
#                  bone boundary where sparse-view methods produce streak artifacts.
#   Checkerboard : 64×64 crop centred on a 4-square corner junction — the sharpest
#                  step edge in the image, where blurring is most visible.
#   DICOM        : 64×64 crop of the right-side skull–soft-tissue boundary — the
#                  most clinically relevant bone edge in the CT_small slice.

ROI_CROPS = {
    'Shepp-Logan'  : (80,  160, 80,  160),   # (y0, y1, x0, x1)
    'Checkerboard' : (96,  160, 96,  160),
    'DICOM'        : (60,  124, 170, 234),
}

print("ROI crop coordinates defined.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import fftpack

# --- Helper Function for Frequency Plot ---
def get_radial_profile(data, center):
    y, x = np.indices(data.shape)
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    r = r.astype(int)
    tbin = np.bincount(r.ravel(), data.ravel())
    nr = np.bincount(r.ravel())
    radialprofile = tbin / np.maximum(nr, 1) # Avoid division by zero
    return radialprofile

# --- Main 10-Panel Plotting Function ---
def plot_gaussian_experiment(title, image, angles, recon,
                             val_psnr, val_ssim, exec_time,
                             gaussians=None,
                             loss_curve=None, psnr_curve=None,
                             dataset_name=None, line_row_idx=None):
    """
    10-panel figure (2x5 grid) for the final report:

    Row 1 (Visual / Process):
      [0,0] Ground Truth      [0,1] Acquisition Geometry   [0,2] Sinogram
      [0,3] Reconstruction    [0,4] ROI Crop Comparison

    Row 2 (Analytical / Metrics):
      [1,0] Residual Map      [1,1] Gaussian Point Cloud   [1,2] 1D Intensity Profile
      [1,3] Frequency Diff    [1,4] Convergence Curves
    """
    # ── Normalise reconstruction to [0,1] for display ───────────────────
    recon_norm = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    H, W = image.shape

    if line_row_idx is None:
        line_row_idx = H // 2  # Default to the middle horizontal line

    # ── Decide which optional panels to show ────────────────────────────
    show_cloud       = gaussians is not None
    show_roi         = (dataset_name is not None) and (dataset_name in ROI_CROPS)
    show_convergence = loss_curve is not None

    # ── Setup 2x5 Grid ──────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 5, figsize=(28, 11))
    fig.subplots_adjust(hspace=0.3, wspace=0.3)

    # Explicitly name axes to prevent indexing bugs
    ax_gt, ax_geom, ax_sino, ax_recon, ax_roi = axes[0]
    ax_err, ax_cloud, ax_line, ax_freq, ax_conv = axes[1]

    fig.suptitle(
        f"{title}\n"
        f"PSNR: {val_psnr:.2f} dB  |  SSIM: {val_ssim:.4f}  |  Time: {exec_time:.1f}s",
        fontsize=16, fontweight='bold', y=0.98
    )

    # ════════════════════════════════════════════════════════════════════
    # ROW 1 (Visual & Process)
    # ════════════════════════════════════════════════════════════════════

    # ── Panel 0,0: Ground Truth ─────────────────────────────────────────
    ax_gt.set_title("Ground Truth", fontweight='bold')
    ax_gt.imshow(image, cmap='gray', vmin=0, vmax=1)
    # Draw faint line to show where the 1D profile is taken
    ax_gt.axhline(y=line_row_idx, color='cyan', linestyle=':', linewidth=1.5, alpha=0.7)
    ax_gt.axis('off')

    # ── Panel 0,1: Acquisition Geometry ─────────────────────────────────
    ax_geom.set_title("Acquisition Geometry", fontweight='bold')
    ax_geom.add_artist(plt.Circle((0, 0), 1, color='lightgray',
                                  fill=False, linestyle='--', linewidth=0.8))
    for theta in angles:
        ax_geom.plot([0, np.cos(theta)], [0, np.sin(theta)],
                     color='steelblue', alpha=0.5, linewidth=1.0)
    ax_geom.set_xlim(-1.1, 1.1)
    ax_geom.set_ylim(-0.1, 1.1)
    ax_geom.set_aspect('equal')
    ax_geom.axis('off')

    # ── Panel 0,2: Sinogram ─────────────────────────────────────────────
    # Assumes generate_sinogram, ANGLES_FULL, DETECTOR_COUNT, full_angles_ref are globally defined
    sino_pred_np = generate_sinogram(recon_norm, angles)
    padded_sino  = np.zeros((ANGLES_FULL, DETECTOR_COUNT))
    for i, a in enumerate(angles):
        idx = np.abs(full_angles_ref - a).argmin()
        padded_sino[idx, :] = sino_pred_np[i, :]
    ax_sino.set_title("Sinogram", fontweight='bold')
    ax_sino.imshow(padded_sino, cmap='gray', aspect='auto')
    ax_sino.set_xlabel("Detector pixels")
    ax_sino.set_ylabel("Angle index (180°)")

    # ── Panel 0,3: Reconstruction ───────────────────────────────────────
    ax_recon.set_title("Reconstruction", fontweight='bold')
    ax_recon.imshow(recon_norm, cmap='gray', vmin=0, vmax=1)
    ax_recon.axis('off')

    # ── Panel 0,4: ROI Comparison (GT vs Recon, zoomed) ─────────────────
    if show_roi:
        y0, y1, x0, x1 = ROI_CROPS[dataset_name]

        gt_crop    = image[y0:y1, x0:x1]
        recon_crop = recon_norm[y0:y1, x0:x1]
        cw = gt_crop.shape[1]

        combined = np.concatenate([gt_crop, recon_crop], axis=1)  # (H, 2W)
        ax_roi.imshow(combined, cmap='gray', vmin=0, vmax=1)
        ax_roi.axvline(x=cw - 0.5, color='yellow', linewidth=1.5, linestyle='--')
        ax_roi.text(cw * 0.25, -3, 'GT', color='yellow', ha='center', fontsize=10, fontweight='bold')
        ax_roi.text(cw * 1.25, -3, 'Recon', color='yellow', ha='center', fontsize=10, fontweight='bold')
        ax_roi.set_title(f"ROI Zoom [{y0}:{y1}, {x0}:{x1}]", fontweight='bold')
        ax_roi.axis('off')

        # Draw the crop rectangle on Panel 0,0 (GT)
        rect = Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=1.5, edgecolor='red', facecolor='none')
        ax_gt.add_patch(rect)
    else:
        ax_roi.axis('off')


    # ════════════════════════════════════════════════════════════════════
    # ROW 2 (Analytical & Metrics)
    # ════════════════════════════════════════════════════════════════════

    # ── Panel 1,0: Residual / Error Map ─────────────────────────────────
    error_map = np.abs(image - recon_norm)
    vmax_err  = np.percentile(error_map, 99)
    im_err = ax_err.imshow(error_map, cmap='hot', vmin=0, vmax=vmax_err)
    ax_err.set_title("Residual Error Map", fontweight='bold')
    ax_err.axis('off')
    fig.colorbar(im_err, ax=ax_err, fraction=0.046, pad=0.04, label='|GT − Recon|')

    # ── Panel 1,1: Gaussian Point Cloud ─────────────────────────────────
    if show_cloud:
        mu_np = gaussians.mu.detach().cpu().numpy()          # (N, 2), in [0,1]
        t_np  = gaussians.intensities.detach().cpu().squeeze().numpy()  # (N,)

        ax_cloud.imshow(recon_norm, cmap='gray', vmin=0, vmax=1,
                        alpha=0.4, extent=[0, W, H, 0])      # faint background
        sc = ax_cloud.scatter(
            mu_np[:, 0] * (W - 1),   # x in pixel coords
            mu_np[:, 1] * (H - 1),   # y in pixel coords
            c=t_np, cmap='plasma', s=0.5, alpha=0.6,
            vmin=0, vmax=1
        )
        ax_cloud.set_xlim(0, W); ax_cloud.set_ylim(H, 0)
        ax_cloud.set_title(f"Gaussian Centres (N={mu_np.shape[0]:,})", fontweight='bold')
        ax_cloud.axis('off')
        fig.colorbar(sc, ax=ax_cloud, fraction=0.046, pad=0.04, label='Intensity')
    else:
        ax_cloud.axis('off')

    # ── Panel 1,2: 1D Intensity Line Profile ────────────────────────────
    ax_line.plot(image[line_row_idx, :], label='Ground Truth', color='black', linestyle='--', linewidth=1.5)
    ax_line.plot(recon_norm[line_row_idx, :], label='2DGS Recon', color='crimson', alpha=0.8, linewidth=1.5)
    ax_line.set_title(f"1D Intensity Profile (Row {line_row_idx})", fontweight='bold')
    ax_line.set_xlabel("Pixel Index")
    ax_line.set_ylabel("Normalized Intensity")
    ax_line.legend(fontsize=8)
    ax_line.grid(True, alpha=0.3)

    # ── Panel 1,3: Energy Distribution by Frequency ─────────────────────
    gt_fft = np.abs(fftpack.fftshift(fftpack.fft2(image)))**2
    recon_fft = np.abs(fftpack.fftshift(fftpack.fft2(recon_norm)))**2
    center = (W//2, H//2)

    gt_radial = get_radial_profile(gt_fft, center)
    recon_radial = get_radial_profile(recon_fft, center)

    min_len = min(len(gt_radial), len(recon_radial))
    # Add epsilon to prevent division by zero in flat areas
    diff = (recon_radial[:min_len] - gt_radial[:min_len]) / (gt_radial[:min_len] + 1e-8)
    freqs = np.linspace(0, 1, min_len)

    ax_freq.axhline(0, color='gray', linestyle='--', linewidth=1.0)
    ax_freq.plot(freqs, diff, color='purple', linewidth=1.5)
    ax_freq.set_title("Energy Diff. by Frequency", fontweight='bold')
    ax_freq.set_xlabel("Relative Frequency")
    ax_freq.set_ylabel("Normalized Difference")
    ax_freq.grid(True, alpha=0.3)

    # ── Panel 1,4: Convergence Curves ───────────────────────────────────
    if show_convergence:
        iters = np.arange(len(loss_curve))
        ax_conv.plot(iters, loss_curve, color='crimson', linewidth=1.5, label='L2 Loss')
        ax_conv.set_xlabel("Iteration")
        ax_conv.set_ylabel("L2 Loss", color='crimson')
        ax_conv.tick_params(axis='y', labelcolor='crimson')
        ax_conv.set_title("Convergence", fontweight='bold')
        ax_conv.axvline(x=3000, color='gray', linestyle='--', linewidth=1.2, label='Stage 2 start')

        if psnr_curve is not None:
            ax_cv2 = ax_conv.twinx()
            ax_cv2.plot(iters, psnr_curve, color='steelblue', linewidth=1.5, label='PSNR')
            ax_cv2.set_ylabel("PSNR (dB)", color='steelblue')
            ax_cv2.tick_params(axis='y', labelcolor='steelblue')

        ax_conv.legend(loc='upper right', fontsize=8)
    else:
        ax_conv.axis('off')

    plt.tight_layout()
    plt.show()

print("plot_gaussian_experiment mapped and defined.")


# 2D Gaussian Reconstruction

## Shepp-Logan Phantom

In [ ]:
current_image = datasets['Shepp-Logan']
angle_sets    = get_angle_sets()
print(f"Dataset: Shepp-Logan  shape={current_image.shape}")

### Uniform Sparse View (20 angles)

In [ ]:
angles = angle_sets['Uniform Sparse View']
print(f"Running 2D Gaussian Reconstruction — Shepp-Logan | Uniform Sparse View ({len(angles)} angles)")

recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
    current_image, angles,
    n_gaussians=8000
)
val_psnr, val_ssim = calculate_metrics(current_image, recon)
print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Uniform Sparse View",
    current_image, angles, recon,
    val_psnr, val_ssim, t,
    gaussians=gaussians,
    loss_curve=loss_curve,
    psnr_curve=psnr_curve,
    dataset_name='Shepp-Logan',
)

### Random Sparse View (20 random angles)

In [ ]:
angles = angle_sets['Random Sparse View']
print(f"Running 2D Gaussian Reconstruction — Shepp-Logan | Random Sparse View ({len(angles)} angles)")

recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
    current_image, angles,
    n_gaussians=8000
)
val_psnr, val_ssim = calculate_metrics(current_image, recon)
print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Random Sparse View",
    current_image, angles, recon,
    val_psnr, val_ssim, t,
    gaussians=gaussians,
    loss_curve=loss_curve,
    psnr_curve=psnr_curve,
    dataset_name='Shepp-Logan',
)

### Missing Wedge (120 angles, ±60° gap)

In [ ]:
angles = angle_sets['Missing Wedge']
print(f"Running 2D Gaussian Reconstruction — Shepp-Logan | Missing Wedge ({len(angles)} angles)")

recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
    current_image, angles,
    n_gaussians=8000
)
val_psnr, val_ssim = calculate_metrics(current_image, recon)
print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
plot_gaussian_experiment(
    "2D Gaussian: Shepp-Logan | Missing Wedge",
    current_image, angles, recon,
    val_psnr, val_ssim, t,
    gaussians=gaussians,
    loss_curve=loss_curve,
    psnr_curve=psnr_curve,
    dataset_name='Shepp-Logan',
)

## Checkerboard

In [ ]:
# current_image = datasets['Checkerboard']
# print(f"Dataset: Checkerboard  shape={current_image.shape}")

### Uniform Sparse View (20 angles)

In [ ]:
# angles = angle_sets['Uniform Sparse View']
# print(f"Running 2D Gaussian Reconstruction — Checkerboard | Uniform Sparse View ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: Checkerboard | Uniform Sparse View",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='Checkerboard',
# )

### Random Sparse View (20 random angles)

In [ ]:
# angles = angle_sets['Random Sparse View']
# print(f"Running 2D Gaussian Reconstruction — Checkerboard | Random Sparse View ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: Checkerboard | Random Sparse View",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='Checkerboard',
# )

### Missing Wedge (120 angles, ±60° gap)

In [ ]:
# angles = angle_sets['Missing Wedge']
# print(f"Running 2D Gaussian Reconstruction — Checkerboard | Missing Wedge ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: Checkerboard | Missing Wedge",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='Checkerboard',
# )

## DICOM

In [ ]:
# current_image = datasets['DICOM']
# print(f"Dataset: DICOM  shape={current_image.shape}")

### Uniform Sparse View (20 angles)

In [ ]:
# angles = angle_sets['Uniform Sparse View']
# print(f"Running 2D Gaussian Reconstruction — DICOM | Uniform Sparse View ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: DICOM | Uniform Sparse View",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='DICOM',
# )

### Random Sparse View (20 random angles)

In [ ]:
# angles = angle_sets['Random Sparse View']
# print(f"Running 2D Gaussian Reconstruction — DICOM | Random Sparse View ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: DICOM | Random Sparse View",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='DICOM',
# )

### Missing Wedge (120 angles, ±60° gap)

In [ ]:
# angles = angle_sets['Missing Wedge']
# print(f"Running 2D Gaussian Reconstruction — DICOM | Missing Wedge ({len(angles)} angles)")

# recon, loss_curve, psnr_curve, t, gaussians = train_gaussian_reconstruction(
#     current_image, angles,
#     n_gaussians=8000, n_iter_stage1=3000, n_iter_stage2=5000,
# )
# val_psnr, val_ssim = calculate_metrics(current_image, recon)
# print(f"\nFinal  PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  Time={t:.1f}s")

In [ ]:
# plot_gaussian_experiment(
#     "2D Gaussian: DICOM | Missing Wedge",
#     current_image, angles, recon,
#     val_psnr, val_ssim, t,
#     gaussians=gaussians,
#     loss_curve=loss_curve,
#     psnr_curve=psnr_curve,
#     dataset_name='DICOM',
# )